# Delta hedging on simulated stock paths

Signed hedging error is terminal financed P&L per 100 option units. Absolute error measures distance from zero. MAE is its average and RMSE weights large errors more heavily. P05 is the fifth percentile of signed P&L.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

root = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'modules' / 'engine.py').is_file()), None)
if root is None:
    raise RuntimeError('Start Jupyter from the VolatilityLab repository.')
sys.path.insert(0, str(root))
from modules.engine import historical_nvda_prices, historical_episode, simulate_path, hedge

def summarize(results):
    return (results.groupby(['Fee case', 'Strategy'], sort=False)['Signed error ($)']
            .agg(Episodes='size', Mean='mean', MAE=lambda x: x.abs().mean(),
                 RMSE=lambda x: np.sqrt(x.pow(2).mean()),
                 P05=lambda x: x.quantile(.05)).round(2))

def review(results, baseline):
    display(summarize(results))
    net = results.loc[results['Fee case'].eq('Net')].copy()
    net['Absolute error ($)'] = net['Signed error ($)'].abs()
    strategies = net['Strategy'].drop_duplicates().tolist()
    terminal = px.box(
        net, x='Strategy', y='Signed error ($)', color='Strategy',
        category_orders={'Strategy': strategies}, points='outliers',
        title='After-fee terminal P&L by strategy',
    )
    terminal.update_xaxes(
        type='category', tickmode='array', tickvals=strategies,
        ticktext=strategies, tickangle=0, automargin=True, showticklabels=True,
    )
    terminal.update_layout(boxmode='overlay', legend_title_text='Strategy')
    terminal.add_hline(y=0).show()
    px.ecdf(net, x='Absolute error ($)', color='Strategy',
            title='Fraction of episodes within each dollar error threshold').show()

    paired = net.pivot(index='Path', columns='Strategy', values='Absolute error ($)')
    gains = paired.drop(columns=baseline).rsub(paired[baseline], axis=0)
    gain_long = gains.melt(var_name='Strategy', value_name='Improvement ($)')
    px.box(gain_long, x='Strategy', y='Improvement ($)',
           title=f'Paired absolute-error improvement versus {baseline}').update_xaxes(type='category').add_hline(y=0).show()

    metrics = summarize(results).reset_index()
    px.bar(metrics, x='Strategy', y='MAE', color='Fee case', barmode='group',
           text_auto='.2f', hover_data=['RMSE', 'Mean', 'P05', 'Episodes'],
           labels={'MAE': 'Mean absolute hedging error ($)'},
           title='Main result · mean absolute error before and after fees').update_xaxes(
               type='category', tickmode='array', tickvals=metrics['Strategy'].unique(),
               ticktext=metrics['Strategy'].unique(), automargin=True).show()
    print(f'Mean paired improvement versus {baseline}, after fees ($):')
    display(gains.mean().round(2))


## Experiment design

Each strategy sees the same five-day stock path and pays the same opening premium. Market volatility is 35%. The 35% hedge is a volatility oracle with access to that assumption. The 20% and 60% hedges deliberately underestimate and overestimate it.

In [ ]:
MARKET_VOL = .35
PATHS = 1000
COST = .001
rows = []
paths = []
for seed in range(PATHS):
    path = simulate_path(MARKET_VOL, seed=seed)
    paths.append(path)
    for vol in (.20, .35, .60):
        for case, fee in [('Net', COST), ('Gross', 0)]:
            result = hedge(path, 100, 5/252, .04, vol, MARKET_VOL, cost=fee)
            rows.append({'Path': seed, 'Strategy': '35% volatility (oracle)' if vol == MARKET_VOL else f'{vol:.0%} volatility', 'Fee case': case,
                         'Signed error ($)': result.PortfolioValue.iloc[-1]})
simulated = pd.DataFrame(rows)


## Explore the simulated prices

The path chart shows the first 30 seeds. The return histogram uses all paths. The final-price histogram shows where the call finishes relative to its strike.

In [ ]:
prices = pd.DataFrame(np.asarray(paths).T).rename_axis('Day')
px.line(prices.iloc[:, :30], labels={'value': 'Stock price ($)', 'variable': 'Seed'},
        title='First 30 simulated stock paths').update_layout(showlegend=False).show()
returns = np.diff(np.log(np.asarray(paths)), axis=1).ravel() * 100
px.histogram(x=returns, nbins=50, labels={'x': 'Daily log return (%)'},
             title='Daily stock returns across all paths').show()
px.histogram(x=prices.iloc[-1], nbins=40, labels={'x': 'Day-five stock price ($)'},
             title='Terminal stock prices and the $100 strike').add_vline(x=100).show()


## Compare hedging errors

The conclusion chart compares mean absolute error before and after fees. Lower is better for tracking zero. In the paired chart, positive values mean a strategy beat the 35% control on the same path.

In [ ]:
review(simulated, '35% volatility (oracle)')

Compare the MAE ranking with the P05 ranking to see whether smaller typical errors also improve downside. These results describe the chosen simulated market, option, and rebalance schedule.